# 04. Advanced Neural Models (Graph Attention Networks)

**Objective:** Implement a Graph Attention Network (GAT) using PyTorch Geometric to capture semantic risk propagation between vulnerabilities. The model learns to route threat intelligence from known critical vulnerabilities to new, unrated ones based on semantic similarity.

**Inputs:** Tabular features (for targets) and raw embeddings (as node features) from `02_feature_engineering`.
**Outputs:** Trained PyTorch GAT model, inductive graph datasets, and Ranking Evaluation.

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, knn_graph
from sklearn.metrics import ndcg_score, roc_auc_score

# Paths
DATA_DIR = os.path.join("..", "Data", "TCTR_features")
MODEL_DIR = os.path.join("..", "models", "TCTR")
os.makedirs(MODEL_DIR, exist_ok=True)

# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
print("Loading feature sets and embeddings...")

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Load tabular features
train_df = pd.read_parquet(os.path.join(DATA_DIR, "train_features.parquet"))
val_df = pd.read_parquet(os.path.join(DATA_DIR, "val_features.parquet"))
test_df = pd.read_parquet(os.path.join(DATA_DIR, "test_features.parquet"))

# Load embeddings
train_emb = torch.tensor(np.load(os.path.join(DATA_DIR, "train_embeddings.npy")), dtype=torch.float)
val_emb = torch.tensor(np.load(os.path.join(DATA_DIR, "val_embeddings.npy")), dtype=torch.float)
test_emb = torch.tensor(np.load(os.path.join(DATA_DIR, "test_embeddings.npy")), dtype=torch.float)

def get_target_tensor(df, random_seed):
    np.random.seed(random_seed)
    base_score = df.get('base_score', pd.Series(5.0, index=df.index)).fillna(5.0)
    velocity = df.get('mock_threat_velocity', pd.Series(0, index=df.index)).fillna(0)
    centrality = df.get('semantic_centrality', pd.Series(0, index=df.index)).fillna(0)
    
    noise = np.random.normal(loc=0.0, scale=3.0, size=len(df))
    raw_risk = (base_score * 0.4) + (np.log1p(np.abs(velocity)) * 8) + (centrality * 4) + noise
    raw_risk = raw_risk.fillna(0)
    
    relevance = pd.qcut(raw_risk, q=5, duplicates='drop', labels=False)
    relevance = pd.Series(relevance).fillna(0).astype(float)
    return torch.tensor(relevance.values, dtype=torch.float).view(-1, 1)

# Extract targets (Y)
y_train = get_target_tensor(train_df, 101)
y_val = get_target_tensor(val_df, 102)
y_test = get_target_tensor(test_df, 103)

# --- NEW: Extract and Normalize Tabular Features ---
features = [
    'desc_length', 'num_keywords', 'num_platforms', 'num_affected_products',
    'days_since_pub_at_horizon', 'days_to_last_modify', 
    'mock_threat_velocity', 'mock_threat_acceleration', 'semantic_centrality'
]
if 'base_score' in train_df.columns:
    features.append('base_score')

scaler = StandardScaler()
X_train_tab = torch.tensor(scaler.fit_transform(train_df[features].fillna(0)), dtype=torch.float)
X_val_tab = torch.tensor(scaler.transform(val_df[features].fillna(0)), dtype=torch.float)
X_test_tab = torch.tensor(scaler.transform(test_df[features].fillna(0)), dtype=torch.float)

# Concatenate embeddings (384) + tabular features (10) to create a rich node feature vector
train_x = torch.cat([train_emb, X_train_tab], dim=1)
val_x = torch.cat([val_emb, X_val_tab], dim=1)
test_x = torch.cat([test_emb, X_test_tab], dim=1)

def build_knn_graph_sklearn(x_full, embeddings_for_knn, y, k=5):
    """Builds graph edges using text embeddings, but assigns the full concatenated features to nodes."""
    print(f"Building KNN graph for {x_full.size(0)} nodes...")
    emb_np = embeddings_for_knn.cpu().numpy()
    
    nbrs = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute', n_jobs=-1).fit(emb_np)
    distances, indices = nbrs.kneighbors(emb_np)
    
    source_nodes, target_nodes = [], []
    for i in range(len(emb_np)):
        for j in range(1, k+1): 
            source_nodes.append(indices[i][j])
            target_nodes.append(i)
            
    edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
    return Data(x=x_full, edge_index=edge_index, y=y).to(device)

# Build Inductive Graphs
train_data = build_knn_graph_sklearn(train_x, train_emb, y_train)
val_data = build_knn_graph_sklearn(val_x, val_emb, y_val)
test_data = build_knn_graph_sklearn(test_x, test_emb, y_test)

print(f"\nNode Feature Dimension: {train_data.x.size(1)} (384 Text + {len(features)} Tabular)")

Loading feature sets and embeddings...
Building KNN graph for 90566 nodes...
Building KNN graph for 40484 nodes...
Building KNN graph for 6976 nodes...

Node Feature Dimension: 394 (384 Text + 10 Tabular)


In [3]:
class ThreatGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super(ThreatGAT, self).__init__()
        # First Graph Attention Layer
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.3)
        # Second Graph Attention Layer 
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False, dropout=0.3)
        # Final linear projection
        self.lin = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.4, training=self.training)
        
        x = F.elu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.4, training=self.training)
        
        out = self.lin(x)
        return out

# Initialize model dynamically picking up the new feature dimension
input_dim = train_data.x.size(1) 
model = ThreatGAT(in_channels=input_dim, hidden_channels=64, out_channels=1).to(device)
print(model)
print(f"Model initialized with input dimension: {input_dim}")

ThreatGAT(
  (conv1): GATConv(394, 64, heads=4)
  (conv2): GATConv(256, 64, heads=1)
  (lin): Linear(in_features=64, out_features=1, bias=True)
)
Model initialized with input dimension: 394


In [4]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.notebook import tqdm

# 1. Use SmoothL1Loss (Huber Loss) which is highly robust to the noise in our targets
criterion = nn.SmoothL1Loss() 

# 2. Lower the initial learning rate and increase weight decay to prevent overfitting
optimizer = AdamW(model.parameters(), lr=0.002, weight_decay=5e-4)

# 3. Add a Scheduler WITHOUT the deprecated 'verbose' argument
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

def train_step(data):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out, data.y)
    loss.backward()
    optimizer.step()
    return loss.item()

def eval_step(data):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        loss = criterion(out, data.y)
    return loss.item(), out.cpu().numpy()

epochs = 200 
best_val_loss = float('inf')
patience = 30
patience_counter = 0

print("Starting Fine-Tuned GAT Training...")

epoch_pbar = tqdm(range(1, epochs + 1), desc="Training GAT")

for epoch in epoch_pbar:
    train_loss = train_step(train_data)
    val_loss, _ = eval_step(val_data)
    
    # Update the learning rate scheduler based on validation loss
    scheduler.step(val_loss)
    
    # Extract the current Learning Rate from the optimizer
    current_lr = optimizer.param_groups[0]['lr']
    
    # Update the progress bar dynamically with Loss AND the live Learning Rate
    epoch_pbar.set_postfix({
        'Train Loss': f"{train_loss:.4f}", 
        'Val Loss': f"{val_loss:.4f}",
        'LR': f"{current_lr:.6f}"
    })
    
    # Early Stopping Logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "best_gat.pth"))
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}")
            break

print(f"Training Complete. Best Validation Loss: {best_val_loss:.4f}")


Starting Fine-Tuned GAT Training...


Training GAT:   0%|          | 0/200 [00:00<?, ?it/s]


Early stopping triggered at epoch 49
Training Complete. Best Validation Loss: 0.8212


In [6]:
# Load the best model weights
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "best_gat.pth"), weights_only=True))

# Get predictions on the out-of-time test graph
test_loss, test_preds = eval_step(test_data)
y_test_np = test_data.y.cpu().numpy().flatten()
test_preds = test_preds.flatten()

# Attach predictions AND true relevance targets back to the test dataframe
test_df_eval = test_df.copy()
test_df_eval['pred_score'] = test_preds
test_df_eval['relevance'] = y_test_np  # <-- FIX: Attach the true targets from the tensor
test_df_eval['query_group'] = pd.to_datetime(test_df_eval['published_date'], utc=True).dt.strftime('%Y-%m')

def calculate_ndcg(df, target_col='relevance', k=10):
    ndcg_scores = []
    for name, group in df.groupby('query_group'):
        # Only calculate if there is more than 1 item and multiple relevance levels to sort
        if len(group) > 1 and len(group[target_col].unique()) > 1:
            true_rel = np.asarray([group[target_col].values])
            pred_scores = np.asarray([group['pred_score'].values])
            ndcg_scores.append(ndcg_score(true_rel, pred_scores, k=k))
    return np.mean(ndcg_scores) if ndcg_scores else 0.0

gat_ndcg = calculate_ndcg(test_df_eval, target_col='relevance', k=10)

print("--- GAT Performance on Test Graph (Out-of-Time) ---")
print(f"Test MSE Loss: {test_loss:.4f}")
print(f"GAT NDCG@10:   {gat_ndcg:.4f}")

--- GAT Performance on Test Graph (Out-of-Time) ---
Test MSE Loss: 1.1731
GAT NDCG@10:   0.6345
